# Analysis

This notebook reads cached result files from `results/cache/` and focuses on evaluation and query-level analysis.

Run `experiments.ipynb` first to generate the cache files.

## Setup

In [1]:
from pathlib import Path

import pandas as pd
import pyterrier as pt
from IPython.display import display
from scipy.stats import ttest_rel

from src.analysis import (
    add_query_features,
    best_system_per_query,
    compare_query_sets,
    label_gain,
    summarize_best_systems,
    summarize_labels,
    top_queries,
)
from src.config import (
    CACHE_PREFIX,
    FAST_MODE,
    FB_TERMS,
    FB_VALUES,
    N_TOPICS,
    RERANK_K,
)
from src.pipelines import (
    add_gain_columns,
    evaluate_runs,
    init_pyterrier,
    perquery_comparison,
    sanitize_topics,
    subset_topics,
)

init_pyterrier()
CACHE_DIR = Path("results/cache")

Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


## Load datasets and qrels

In [2]:
# Keep the same dataset loading logic as the experiments notebook.
dl19_topics_ds = pt.get_dataset("irds:msmarco-passage/trec-dl-2019")
dl19_qrels_ds = pt.get_dataset("irds:msmarco-passage/trec-dl-2019/judged")
dl20 = pt.get_dataset("irds:msmarco-passage/trec-dl-2020/judged")

topics_19 = sanitize_topics(dl19_topics_ds.get_topics())
qrels_19 = dl19_qrels_ds.get_qrels()
topics_19 = topics_19[topics_19["qid"].astype(str).isin(qrels_19["qid"].astype(str))].copy()

topics_20 = sanitize_topics(dl20.get_topics())
qrels_20 = dl20.get_qrels()

## Analysis configuration

In [3]:
topics_19_run, qrels_19_run = subset_topics(topics_19, qrels_19, n_topics=N_TOPICS)
topics_20_run, qrels_20_run = subset_topics(topics_20, qrels_20, n_topics=N_TOPICS)

def load_run(filename: str) -> pd.DataFrame:
    return pd.read_pickle(CACHE_DIR / filename)


## Load cached core runs

In [4]:
runs_19 = {
    "BM25": load_run(f"{CACHE_PREFIX}_bm25_dl19.pkl"),
    "BM25+RM3": load_run(f"{CACHE_PREFIX}_bm25_rm3_fb{FB_TERMS}_dl19.pkl"),
    "BM25+TAS-B": load_run(f"{CACHE_PREFIX}_bm25_tasb_k{RERANK_K}_dl19.pkl"),
    "BM25+RM3+TAS-B": load_run(f"{CACHE_PREFIX}_bm25_rm3_fb{FB_TERMS}_tasb_k{RERANK_K}_dl19.pkl"),
}

runs_20 = {
    "BM25": load_run(f"{CACHE_PREFIX}_bm25_dl20.pkl"),
    "BM25+RM3": load_run(f"{CACHE_PREFIX}_bm25_rm3_fb{FB_TERMS}_dl20.pkl"),
    "BM25+TAS-B": load_run(f"{CACHE_PREFIX}_bm25_tasb_k{RERANK_K}_dl20.pkl"),
    "BM25+RM3+TAS-B": load_run(f"{CACHE_PREFIX}_bm25_rm3_fb{FB_TERMS}_tasb_k{RERANK_K}_dl20.pkl"),
}

## Aggregate evaluation

In [5]:
evaluate_runs(runs_19, topics_19_run, qrels_19_run)

,name,map,recall_100,ndcg_cut_10
0,BM25,0.370004,0.442170,0.479540
1,BM25+RM3,0.406129,0.456336,0.524715
2,BM25+TAS-B,0.359250,0.442210,0.689030
3,BM25+RM3+TAS-B,0.367400,0.456336,0.687443


In [6]:
evaluate_runs(runs_20, topics_20_run, qrels_20_run)

,name,map,recall_100,ndcg_cut_10
0,BM25,0.358724,0.504658,0.493627
1,BM25+RM3,0.400468,0.548754,0.510773
2,BM25+TAS-B,0.385519,0.505561,0.657714
3,BM25+RM3+TAS-B,0.412778,0.549097,0.671603


## Per-query comparison for the core systems

In [7]:
perquery_19 = perquery_comparison(runs_19, topics_19_run, qrels_19_run)
perquery_19 = add_gain_columns(
    perquery_19,
    baseline="BM25",
    systems=["BM25+RM3", "BM25+TAS-B", "BM25+RM3+TAS-B"],
)
perquery_19.sort_values("BM25+RM3+TAS-B_gain", ascending=False).head(10)

,qid,query,BM25,BM25+RM3,BM25+RM3+TAS-B,BM25+TAS-B,BM25+RM3_gain,BM25+TAS-B_gain,BM25+RM3+TAS-B_gain
25,183378,exons definition biology,0.366266,0.697505,1.000000,1.000000,0.331239,0.633734,0.633734
11,264014,how long is life cycle of flea,0.334922,0.355459,0.835440,0.845763,0.020536,0.510841,0.500518
18,1115776,what is an aml surveillance analyst,0.334768,0.361401,0.834189,0.766642,0.026633,0.431873,0.499420
35,1114646,what is famvir prescribed for,0.363148,0.358963,0.815633,0.815633,-0.004185,0.452485,0.452485
15,148538,difference between rn and bsn,0.163883,0.263902,0.590363,0.617333,0.100020,0.453450,0.426480
28,490595,rsa definition key,0.223715,0.190264,0.634853,0.634853,-0.033450,0.411138,0.411138
41,1129237,hydrogen is a liquid below what temperature,0.547601,0.656812,0.929698,0.810827,0.109211,0.263225,0.382096
31,443396,lps laws definition,0.069431,0.066254,0.435098,0.384311,-0.003177,0.314880,0.365666
38,405717,is cdg airport in main paris,0.375915,0.340584,0.738404,0.738404,-0.035331,0.362488,0.362488
13,962179,when was the salvation army founded,0.000000,0.000000,0.358519,0.570542,0.000000,0.570542,0.358519


## Statistical test on DL19

Compare `BM25+TAS-B` against `BM25+RM3+TAS-B` on the per-query DL19 scores.

In [8]:
perquery_19["RM3_vs_TASB_gain"] = perquery_19["BM25+RM3+TAS-B"] - perquery_19["BM25+TAS-B"]

t_stat, p_value = ttest_rel(
    perquery_19["BM25+RM3+TAS-B"],
    perquery_19["BM25+TAS-B"],
)

pd.DataFrame([
    {
        "comparison": "BM25+RM3+TAS-B vs BM25+TAS-B",
        "mean_gain": perquery_19["RM3_vs_TASB_gain"].mean(),
        "t_statistic": t_stat,
        "p_value": p_value,
    }
])

,comparison,mean_gain,t_statistic,p_value
0,BM25+RM3+TAS-B vs BM25+TAS-B,-0.001587,-0.195809,0.845704


## Top DL19 queries for RM3 before TAS-B

Top 5 improved and top 5 harmed queries when comparing `BM25+RM3+TAS-B` against `BM25+TAS-B`.

In [9]:
top_5_improved_19 = top_queries(perquery_19, "RM3_vs_TASB_gain", n=5, ascending=False)
top_5_harmed_19 = top_queries(perquery_19, "RM3_vs_TASB_gain", n=5, ascending=True)

print("Top 5 improved")
display(top_5_improved_19)

print("Top 5 harmed")
display(top_5_harmed_19)

Top 5 improved


,qid,query,RM3_vs_TASB_gain,BM25,BM25+RM3,BM25+RM3+TAS-B,BM25+TAS-B,BM25+RM3_gain,BM25+TAS-B_gain,BM25+RM3+TAS-B_gain
41,1129237,hydrogen is a liquid below what temperature,0.118871,0.547601,0.656812,0.929698,0.810827,0.109211,0.263225,0.382096
36,19335,anthropological definition of environment,0.095534,0.441089,0.522705,0.596099,0.500565,0.081616,0.059476,0.155010
19,1112341,what is the daily life of thai people,0.088520,0.587924,0.612293,0.824226,0.735705,0.024368,0.147781,0.236301
18,1115776,what is an aml surveillance analyst,0.067547,0.334768,0.361401,0.834189,0.766642,0.026633,0.431873,0.499420
10,915593,what types of food can you cook sous vide,0.066619,0.417021,0.358954,0.510716,0.444097,-0.058067,0.027076,0.093696


Top 5 harmed


,qid,query,RM3_vs_TASB_gain,BM25,BM25+RM3,BM25+RM3+TAS-B,BM25+TAS-B,BM25+RM3_gain,BM25+TAS-B_gain,BM25+RM3+TAS-B_gain
13,962179,when was the salvation army founded,-0.212023,0.000000,0.000000,0.358519,0.570542,0.000000,0.570542,0.358519
26,1106007,define visceral?,-0.130569,0.368773,0.649195,0.626394,0.756963,0.280422,0.388190,0.257621
8,527433,types of dysarthria from cerebral palsy,-0.082948,0.293011,0.293011,0.620995,0.703943,0.000000,0.410932,0.327984
33,87452,causes of military suicide,-0.051388,0.498614,0.397099,0.488759,0.540147,-0.101515,0.041533,-0.009855
40,1113437,what is physical description of spruce,-0.036094,0.226504,0.117732,0.251215,0.287309,-0.108771,0.060805,0.024712


In [10]:
perquery_20 = perquery_comparison(runs_20, topics_20_run, qrels_20_run)
perquery_20 = add_gain_columns(
    perquery_20,
    baseline="BM25",
    systems=["BM25+RM3", "BM25+TAS-B", "BM25+RM3+TAS-B"],
)
perquery_20.sort_values("BM25+RM3+TAS-B_gain", ascending=False).head(10)

,qid,query,BM25,BM25+RM3,BM25+RM3+TAS-B,BM25+TAS-B,BM25+RM3_gain,BM25+TAS-B_gain,BM25+RM3+TAS-B_gain
32,324585,how much money do motivational speakers make,0.069431,0.069431,0.889954,0.889954,0.000000,0.820523,0.820523
41,583468,what carvedilol used for,0.275930,0.257691,0.936808,0.936808,-0.018239,0.660877,0.660877
18,1132532,average annual income data analyst,0.150654,0.176413,0.738222,0.734567,0.025759,0.583913,0.587568
51,938400,when did family feud come out?,0.114339,0.134713,0.655737,0.655737,0.020374,0.541398,0.541398
17,1131069,how many sons robert kraft has,0.469279,0.758587,1.000000,1.000000,0.289308,0.530721,0.530721
53,997622,where is the show shameless filmed,0.481900,0.710023,1.000000,0.955831,0.228123,0.473931,0.518100
52,940547,when did rock n roll begin?,0.127349,0.127349,0.611993,0.547782,0.000000,0.420433,0.484643
35,336901,how old is vanessa redgrave,0.156426,0.167160,0.625705,0.617320,0.010734,0.460893,0.469279
10,1110678,what is the un fao,0.332921,0.433885,0.747478,0.666948,0.100964,0.334027,0.414557
1,1037496,who is rep scalise?,0.542564,0.522524,0.927449,0.885036,-0.020039,0.342472,0.384886


## Statistical test on DL20

Compare `BM25+TAS-B` against `BM25+RM3+TAS-B` on the per-query DL20 scores.

In [11]:
perquery_20["RM3_vs_TASB_gain"] = perquery_20["BM25+RM3+TAS-B"] - perquery_20["BM25+TAS-B"]

t_stat, p_value = ttest_rel(
    perquery_20["BM25+RM3+TAS-B"],
    perquery_20["BM25+TAS-B"],
)

pd.DataFrame([
    {
        "comparison": "BM25+RM3+TAS-B vs BM25+TAS-B",
        "mean_gain": perquery_20["RM3_vs_TASB_gain"].mean(),
        "t_statistic": t_stat,
        "p_value": p_value,
    }
])

,comparison,mean_gain,t_statistic,p_value
0,BM25+RM3+TAS-B vs BM25+TAS-B,0.013889,1.470082,0.147452


## Top DL20 queries for RM3 before TAS-B

Top 5 improved and top 5 harmed queries when comparing `BM25+RM3+TAS-B` against `BM25+TAS-B`.

In [12]:
top_5_improved_20 = top_queries(perquery_20, "RM3_vs_TASB_gain", n=5, ascending=False)
top_5_harmed_20 = top_queries(perquery_20, "RM3_vs_TASB_gain", n=5, ascending=True)

print("Top 5 improved")
display(top_5_improved_20)

print("Top 5 harmed")
display(top_5_harmed_20)

Top 5 improved


,qid,query,RM3_vs_TASB_gain,BM25,BM25+RM3,BM25+RM3+TAS-B,BM25+TAS-B,BM25+RM3_gain,BM25+TAS-B_gain,BM25+RM3+TAS-B_gain
14,1121353,what can you do about discrimination in the wo...,0.270007,0.725837,0.755647,0.825453,0.555447,0.029811,-0.170390,0.099617
29,174463,dog day afternoon meaning,0.261840,0.426388,0.496178,0.582539,0.320699,0.069790,-0.105689,0.156150
8,1108651,what the best way to get clothes white,0.213313,0.292512,0.345315,0.644030,0.430717,0.052803,0.138205,0.351518
10,1110678,what is the un fao,0.080530,0.332921,0.433885,0.747478,0.666948,0.100964,0.334027,0.414557
52,940547,when did rock n roll begin?,0.064210,0.127349,0.127349,0.611993,0.547782,0.000000,0.420433,0.484643


Top 5 harmed


,qid,query,RM3_vs_TASB_gain,BM25,BM25+RM3,BM25+RM3+TAS-B,BM25+TAS-B,BM25+RM3_gain,BM25+TAS-B_gain,BM25+RM3+TAS-B_gain
3,1051399,who sings monk theme song,-0.234054,0.179623,0.000000,0.563687,0.797741,-0.179623,0.618118,0.384064
42,640502,what does it mean if your tsh is low,-0.051279,0.541818,0.552214,0.863015,0.914294,0.010396,0.372476,0.321197
2,1043135,who killed nicholas ii of russia,-0.038074,0.595064,0.600326,0.505473,0.543548,0.005262,-0.051516,-0.089591
43,67316,can fever cause miscarriage early pregnancy,-0.028745,0.242096,0.152745,0.293011,0.321756,-0.089350,0.079661,0.050915
28,169208,does mississippi have an income tax,0.000000,0.471107,0.599517,0.693266,0.693266,0.128409,0.222159,0.222159


## Inspect a single query

In [13]:
example_qid = str(perquery_19.iloc[0]["qid"])
perquery_19[perquery_19["qid"].astype(str) == example_qid]

,qid,query,BM25,BM25+RM3,BM25+RM3+TAS-B,BM25+TAS-B,BM25+RM3_gain,BM25+TAS-B_gain,BM25+RM3+TAS-B_gain,RM3_vs_TASB_gain
0,156493,do goldfish grow,0.963412,1.0,0.900864,0.900864,0.036588,-0.062548,-0.062548,0.0


## Load RM3 + TAS-B sweep caches

In [14]:
rm3_tasb_runs_19 = {"BM25+TAS-B": runs_19["BM25+TAS-B"]}
rm3_tasb_runs_20 = {"BM25+TAS-B": runs_20["BM25+TAS-B"]}

for fb in FB_VALUES:
    rm3_tasb_runs_19[f"BM25+RM3(fb={fb})+TAS-B"] = load_run(
        f"{CACHE_PREFIX}_bm25_rm3_fb{fb}_tasb_k{RERANK_K}_dl19.pkl"
    )
    rm3_tasb_runs_20[f"BM25+RM3(fb={fb})+TAS-B"] = load_run(
        f"{CACHE_PREFIX}_bm25_rm3_fb{fb}_tasb_k{RERANK_K}_dl20.pkl"
    )

## RM3 analysis on DL19

In [15]:
evaluate_runs(rm3_tasb_runs_19, topics_19_run, qrels_19_run)

,name,map,recall_100,ndcg_cut_10
0,BM25+TAS-B,0.359250,0.442210,0.689030
1,BM25+RM3(fb=5)+TAS-B,0.364385,0.454594,0.686137
2,BM25+RM3(fb=10)+TAS-B,0.367400,0.456336,0.687443
3,BM25+RM3(fb=20)+TAS-B,0.372978,0.465349,0.688454
4,BM25+RM3(fb=30)+TAS-B,0.372421,0.464965,0.688710
5,BM25+RM3(fb=50)+TAS-B,0.373623,0.465664,0.688618


In [16]:
rm3_perquery_19 = perquery_comparison(rm3_tasb_runs_19, topics_19_run, qrels_19_run)
rm3_analysis_19 = add_gain_columns(
    rm3_perquery_19,
    baseline="BM25+TAS-B",
    systems=[f"BM25+RM3(fb={fb})+TAS-B" for fb in FB_VALUES],
)
rm3_analysis_19 = add_query_features(rm3_analysis_19)
rm3_analysis_19 = label_gain(
    rm3_analysis_19,
    gain_column=f"BM25+RM3(fb={FB_TERMS})+TAS-B_gain",
)
summarize_labels(rm3_analysis_19, f"BM25+RM3(fb={FB_TERMS})+TAS-B_gain_label")

,BM25+RM3(fb=10)+TAS-B_gain_label,count
0,neutral,27
1,hurt,9
2,helped,7


In [17]:
top_queries(rm3_analysis_19, f"BM25+RM3(fb={FB_TERMS})+TAS-B_gain", n=10, ascending=False)

,qid,query,BM25+RM3(fb=10)+TAS-B_gain,BM25+RM3(fb=10)+TAS-B,BM25+RM3(fb=20)+TAS-B,BM25+RM3(fb=30)+TAS-B,BM25+RM3(fb=5)+TAS-B,BM25+RM3(fb=50)+TAS-B,BM25+TAS-B,BM25+RM3(fb=5)+TAS-B_gain,BM25+RM3(fb=20)+TAS-B_gain,BM25+RM3(fb=30)+TAS-B_gain,BM25+RM3(fb=50)+TAS-B_gain,query_len_words,query_len_chars,has_colon,has_digit,is_question
41,1129237,hydrogen is a liquid below what temperature,0.118871,0.929698,0.929698,0.929698,0.929698,0.929698,0.810827,0.118871,0.118871,0.118871,0.118871,7,43,False,False,False
36,19335,anthropological definition of environment,0.095534,0.596099,0.596099,0.596099,0.565071,0.596099,0.500565,0.064506,0.095534,0.095534,0.095534,4,41,False,False,False
19,1112341,what is the daily life of thai people,0.088520,0.824226,0.803019,0.803019,0.824226,0.803019,0.735705,0.088520,0.067313,0.067313,0.067313,8,37,False,False,False
18,1115776,what is an aml surveillance analyst,0.067547,0.834189,0.766642,0.766642,0.801100,0.766642,0.766642,0.034458,0.000000,0.000000,0.000000,6,35,False,False,False
10,915593,what types of food can you cook sous vide,0.066619,0.510716,0.600470,0.600470,0.444097,0.600470,0.444097,0.000000,0.156373,0.156373,0.156373,9,41,False,False,False
31,443396,lps laws definition,0.050786,0.435098,0.412692,0.388244,0.435098,0.384311,0.384311,0.050786,0.028381,0.003933,0.000000,3,19,False,False,False
23,207786,how are some sharks warm blooded,0.036461,0.595740,0.595740,0.595740,0.595740,0.595740,0.559279,0.036461,0.036461,0.036461,0.036461,6,32,False,False,False
7,1133167,how is the weather in jamaica,0.003215,0.698263,0.698263,0.698263,0.698263,0.698263,0.695048,0.003215,0.003215,0.003215,0.003215,6,29,False,False,False
22,833860,what is the most popular food in switzerland,0.001678,0.938863,0.945260,0.945260,0.945260,0.945260,0.937185,0.008075,0.008075,0.008075,0.008075,8,44,False,False,False
32,1121709,what are the three percenters?,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5,30,False,False,True


In [18]:
top_queries(rm3_analysis_19, f"BM25+RM3(fb={FB_TERMS})+TAS-B_gain", n=10, ascending=True)

,qid,query,BM25+RM3(fb=10)+TAS-B_gain,BM25+RM3(fb=10)+TAS-B,BM25+RM3(fb=20)+TAS-B,BM25+RM3(fb=30)+TAS-B,BM25+RM3(fb=5)+TAS-B,BM25+RM3(fb=50)+TAS-B,BM25+TAS-B,BM25+RM3(fb=5)+TAS-B_gain,BM25+RM3(fb=20)+TAS-B_gain,BM25+RM3(fb=30)+TAS-B_gain,BM25+RM3(fb=50)+TAS-B_gain,query_len_words,query_len_chars,has_colon,has_digit,is_question
13,962179,when was the salvation army founded,-0.212023,0.358519,0.358519,0.358519,0.358519,0.358519,0.570542,-0.212023,-0.212023,-0.212023,-0.212023,6,35,False,False,False
26,1106007,define visceral?,-0.130569,0.626394,0.626394,0.626394,0.626394,0.626394,0.756963,-0.130569,-0.130569,-0.130569,-0.130569,2,16,False,False,True
8,527433,types of dysarthria from cerebral palsy,-0.082948,0.620995,0.656124,0.656124,0.630389,0.656124,0.703943,-0.073554,-0.047819,-0.047819,-0.047819,6,39,False,False,False
33,87452,causes of military suicide,-0.051388,0.488759,0.540147,0.540147,0.466674,0.540147,0.540147,-0.073472,0.000000,0.000000,0.000000,4,26,False,False,False
40,1113437,what is physical description of spruce,-0.036094,0.251215,0.198722,0.234191,0.332125,0.234191,0.287309,0.044816,-0.088587,-0.053118,-0.053118,6,38,False,False,False
15,148538,difference between rn and bsn,-0.026970,0.590363,0.590363,0.590363,0.590363,0.590363,0.617333,-0.026970,-0.026970,-0.026970,-0.026970,5,29,False,False,False
24,1114819,what is durable medical equipment consist of,-0.024455,0.788901,0.813356,0.813356,0.788901,0.813356,0.813356,-0.024455,0.000000,0.000000,0.000000,7,44,False,False,False
37,47923,axon terminals or synaptic knob definition,-0.022716,0.542444,0.542444,0.542444,0.542444,0.542444,0.565161,-0.022716,-0.022716,-0.022716,-0.022716,6,42,False,False,False
11,264014,how long is life cycle of flea,-0.010323,0.835440,0.835440,0.835440,0.835440,0.835440,0.845763,-0.010323,-0.010323,-0.010323,-0.010323,7,30,False,False,False
27,1124210,tracheids are part of _____.,0.000000,0.784233,0.784233,0.784233,0.784233,0.784233,0.784233,0.000000,0.000000,0.000000,0.000000,5,28,False,False,False


In [19]:
compare_query_sets(
    rm3_analysis_19,
    label_column=f"BM25+RM3(fb={FB_TERMS})+TAS-B_gain_label",
    feature_columns=["query_len_words", "query_len_chars", "has_colon", "has_digit", "is_question"],
)

,BM25+RM3(fb=10)+TAS-B_gain_label,query_len_words,query_len_chars,has_colon,has_digit,is_question
0,helped,6.142857,35.428571,0.0,0.000000,0.000000
1,hurt,5.444444,33.222222,0.0,0.000000,0.111111
2,neutral,5.185185,31.888889,0.0,0.037037,0.037037


In [20]:
rm3_metric_cols_19 = [f"BM25+RM3(fb={fb})+TAS-B" for fb in FB_VALUES]
rm3_gain_cols_19 = [f"BM25+RM3(fb={fb})+TAS-B_gain" for fb in FB_VALUES]
rm3_analysis_19[rm3_metric_cols_19].mean().sort_values(ascending=False)

BM25+RM3(fb=30)+TAS-B    0.688710
BM25+RM3(fb=50)+TAS-B    0.688618
BM25+RM3(fb=20)+TAS-B    0.688454
BM25+RM3(fb=10)+TAS-B    0.687443
BM25+RM3(fb=5)+TAS-B     0.686137
dtype: float64

In [21]:
rm3_best_19 = best_system_per_query(rm3_analysis_19, rm3_metric_cols_19)
summarize_best_systems(rm3_best_19)

,best_system,count
0,BM25+RM3(fb=5)+TAS-B,37
1,BM25+RM3(fb=20)+TAS-B,4
2,BM25+RM3(fb=10)+TAS-B,2


In [22]:
rm3_best_gain_19 = best_system_per_query(rm3_analysis_19, rm3_gain_cols_19, output_column="best_gain_setting")
summarize_best_systems(rm3_best_gain_19, best_system_column="best_gain_setting")

,best_gain_setting,count
0,BM25+RM3(fb=5)+TAS-B_gain,37
1,BM25+RM3(fb=20)+TAS-B_gain,4
2,BM25+RM3(fb=10)+TAS-B_gain,2


## Is the RM3 effect consistent per query on DL19?

Compare the gain from RM3 on the lexical pipeline with the gain from RM3 before TAS-B.

In [23]:
rm3_consistency_19 = perquery_19[["qid", "query", "BM25+RM3_gain"]].copy()
rm3_consistency_19["BM25+RM3+TAS-B_gain_vs_TAS-B"] = perquery_19["RM3_vs_TASB_gain"]

rm3_consistency_19["gain_rm3_bm25"] = rm3_consistency_19["BM25+RM3_gain"]
rm3_consistency_19["gain_rm3_rerank"] = rm3_consistency_19["BM25+RM3+TAS-B_gain_vs_TAS-B"]

rm3_consistency_19[["gain_rm3_bm25", "gain_rm3_rerank"]].corr()

,gain_rm3_bm25,gain_rm3_rerank
gain_rm3_bm25,1.000000,-0.049081
gain_rm3_rerank,-0.049081,1.000000


In [24]:
def gain_direction(value: float, threshold: float = 0.01) -> str:
    if value >= threshold:
        return "helped"
    if value <= -threshold:
        return "hurt"
    return "neutral"

rm3_consistency_19["bm25_effect"] = rm3_consistency_19["gain_rm3_bm25"].apply(gain_direction)
rm3_consistency_19["rerank_effect"] = rm3_consistency_19["gain_rm3_rerank"].apply(gain_direction)
rm3_consistency_19["consistency_group"] = (
    rm3_consistency_19["bm25_effect"] + " -> " + rm3_consistency_19["rerank_effect"]
)

rm3_consistency_19["consistency_group"].value_counts().rename_axis("consistency_group").reset_index(name="count")

,consistency_group,count
0,helped -> neutral,15
1,hurt -> neutral,6
2,neutral -> neutral,6
3,helped -> hurt,5
4,helped -> helped,4
5,neutral -> hurt,2
6,hurt -> helped,2
7,hurt -> hurt,2
8,neutral -> helped,1


In [25]:
rm3_consistency_19.sort_values("gain_rm3_bm25", ascending=False)[[
    "qid",
    "query",
    "gain_rm3_bm25",
    "gain_rm3_rerank",
    "consistency_group",
]].head(10)

,qid,query,gain_rm3_bm25,gain_rm3_rerank,consistency_group
25,183378,exons definition biology,0.331239,0.000000,helped -> neutral
26,1106007,define visceral?,0.280422,-0.130569,helped -> hurt
14,1117099,what is a active margin,0.262498,0.000000,helped -> neutral
22,833860,what is the most popular food in switzerland,0.240082,0.001678,helped -> neutral
21,104861,cost of interior concrete flooring,0.223694,0.000000,helped -> neutral
29,1103812,who formed the commonwealth of independent states,0.211811,0.000000,helped -> neutral
24,1114819,what is durable medical equipment consist of,0.142904,-0.024455,helped -> hurt
12,1121402,what can contour plowing reduce,0.122385,0.000000,helped -> neutral
41,1129237,hydrogen is a liquid below what temperature,0.109211,0.118871,helped -> helped
15,148538,difference between rn and bsn,0.100020,-0.026970,helped -> hurt


## `fb_terms` comparison on DL19

Compare the different `fb_terms` settings using `ndcg_cut_10` and `map`.

In [26]:
fb_terms_eval_19 = evaluate_runs(
    rm3_tasb_runs_19,
    topics_19_run,
    qrels_19_run,
    eval_metrics=["ndcg_cut_10", "map"],
)

fb_terms_table_19 = fb_terms_eval_19[fb_terms_eval_19["name"] != "BM25+TAS-B"].copy()
fb_terms_table_19["fb_terms"] = fb_terms_table_19["name"].str.extract(r"fb=(\d+)").astype(int)
fb_terms_table_19 = fb_terms_table_19[["fb_terms", "ndcg_cut_10", "map"]].sort_values("fb_terms")
fb_terms_table_19

,fb_terms,ndcg_cut_10,map
1,5,0.686137,0.364385
2,10,0.687443,0.367400
3,20,0.688454,0.372978
4,30,0.688710,0.372421
5,50,0.688618,0.373623


## BM25 + RM3 recall analysis on DL19

In [27]:
bm25_rm3_runs_19 = {"BM25": runs_19["BM25"]}

for fb in FB_VALUES:
    bm25_rm3_runs_19[f"BM25+RM3(fb={fb})"] = load_run(
        f"{CACHE_PREFIX}_bm25_rm3_fb{fb}_dl19.pkl"
    )

bm25_rm3_perquery_19_recall = perquery_comparison(
    bm25_rm3_runs_19,
    topics_19_run,
    qrels_19_run,
    metric="recall_100",
)

bm25_rm3_analysis_19_recall = add_gain_columns(
    bm25_rm3_perquery_19_recall,
    baseline="BM25",
    systems=[f"BM25+RM3(fb={fb})" for fb in FB_VALUES],
)
bm25_rm3_analysis_19_recall = add_query_features(bm25_rm3_analysis_19_recall)

In [28]:
bm25_rm3_metric_cols_19 = [f"BM25+RM3(fb={fb})" for fb in FB_VALUES]
bm25_rm3_gain_cols_19 = [f"BM25+RM3(fb={fb})_gain" for fb in FB_VALUES]
bm25_rm3_analysis_19_recall[bm25_rm3_metric_cols_19].mean().sort_values(ascending=False)

BM25+RM3(fb=50)    0.465664
BM25+RM3(fb=20)    0.465349
BM25+RM3(fb=30)    0.464965
BM25+RM3(fb=10)    0.456336
BM25+RM3(fb=5)     0.454525
dtype: float64

In [29]:
bm25_rm3_best_19 = best_system_per_query(bm25_rm3_analysis_19_recall, bm25_rm3_metric_cols_19)
summarize_best_systems(bm25_rm3_best_19)

,best_system,count
0,BM25+RM3(fb=5),24
1,BM25+RM3(fb=20),11
2,BM25+RM3(fb=10),4
3,BM25+RM3(fb=50),3
4,BM25+RM3(fb=30),1


In [30]:
bm25_rm3_best_gain_19 = best_system_per_query(
    bm25_rm3_analysis_19_recall,
    bm25_rm3_gain_cols_19,
    output_column="best_gain_setting",
)
summarize_best_systems(bm25_rm3_best_gain_19, best_system_column="best_gain_setting")

,best_gain_setting,count
0,BM25+RM3(fb=5)_gain,24
1,BM25+RM3(fb=20)_gain,11
2,BM25+RM3(fb=10)_gain,4
3,BM25+RM3(fb=50)_gain,3
4,BM25+RM3(fb=30)_gain,1


In [31]:
bm25_rm3_analysis_19_recall = label_gain(
    bm25_rm3_analysis_19_recall,
    gain_column=f"BM25+RM3(fb={FB_TERMS})_gain",
)
summarize_labels(bm25_rm3_analysis_19_recall, f"BM25+RM3(fb={FB_TERMS})_gain_label")

,BM25+RM3(fb=10)_gain_label,count
0,neutral,19
1,helped,18
2,hurt,6


In [32]:
top_queries(bm25_rm3_analysis_19_recall, f"BM25+RM3(fb={FB_TERMS})_gain", n=10, ascending=False)

,qid,query,BM25+RM3(fb=10)_gain,BM25,BM25+RM3(fb=10),BM25+RM3(fb=20),BM25+RM3(fb=30),BM25+RM3(fb=5),BM25+RM3(fb=50),BM25+RM3(fb=5)_gain,BM25+RM3(fb=20)_gain,BM25+RM3(fb=30)_gain,BM25+RM3(fb=50)_gain,query_len_words,query_len_chars,has_colon,has_digit,is_question
25,183378,exons definition biology,0.113537,0.235808,0.349345,0.344978,0.349345,0.336245,0.353712,0.100437,0.109170,0.113537,0.117904,3,24,False,False,False
40,1113437,what is physical description of spruce,0.103896,0.272727,0.376623,0.324675,0.324675,0.363636,0.324675,0.090909,0.051948,0.051948,0.051948,6,38,False,False,False
29,1103812,who formed the commonwealth of independent states,0.096774,0.612903,0.709677,0.741935,0.741935,0.709677,0.741935,0.096774,0.129032,0.129032,0.129032,7,49,False,False,False
9,1037798,who is robert gray,0.076923,0.615385,0.692308,0.769231,0.769231,0.692308,0.769231,0.076923,0.153846,0.153846,0.153846,4,18,False,False,False
14,1117099,what is a active margin,0.075630,0.310924,0.386555,0.394958,0.386555,0.378151,0.386555,0.067227,0.084034,0.075630,0.075630,5,23,False,False,False
17,359349,how to find the midsegment of a trapezoid,0.071429,0.500000,0.571429,0.571429,0.589286,0.589286,0.589286,0.089286,0.071429,0.089286,0.089286,8,41,False,False,False
41,1129237,hydrogen is a liquid below what temperature,0.071429,0.678571,0.750000,0.750000,0.750000,0.750000,0.750000,0.071429,0.071429,0.071429,0.071429,7,43,False,False,False
35,1114646,what is famvir prescribed for,0.057692,0.673077,0.730769,0.730769,0.730769,0.750000,0.730769,0.076923,0.057692,0.057692,0.057692,5,29,False,False,False
22,833860,what is the most popular food in switzerland,0.053333,0.226667,0.280000,0.306667,0.280000,0.306667,0.280000,0.080000,0.080000,0.053333,0.053333,8,44,False,False,False
26,1106007,define visceral?,0.050000,0.233333,0.283333,0.266667,0.266667,0.266667,0.266667,0.033333,0.033333,0.033333,0.033333,2,16,False,False,True


## RM3 analysis on DL20

In [33]:
evaluate_runs(rm3_tasb_runs_20, topics_20_run, qrels_20_run)

,name,map,recall_100,ndcg_cut_10
0,BM25+TAS-B,0.385519,0.505561,0.657714
1,BM25+RM3(fb=5)+TAS-B,0.415240,0.552288,0.674037
2,BM25+RM3(fb=10)+TAS-B,0.412778,0.549097,0.671603
3,BM25+RM3(fb=20)+TAS-B,0.414257,0.549762,0.676079
4,BM25+RM3(fb=30)+TAS-B,0.415657,0.551131,0.676006
5,BM25+RM3(fb=50)+TAS-B,0.416002,0.551416,0.675926


In [34]:
rm3_perquery_20 = perquery_comparison(rm3_tasb_runs_20, topics_20_run, qrels_20_run)
rm3_analysis_20 = add_gain_columns(
    rm3_perquery_20,
    baseline="BM25+TAS-B",
    systems=[f"BM25+RM3(fb={fb})+TAS-B" for fb in FB_VALUES],
)
rm3_analysis_20 = add_query_features(rm3_analysis_20)
rm3_analysis_20 = label_gain(
    rm3_analysis_20,
    gain_column=f"BM25+RM3(fb={FB_TERMS})+TAS-B_gain",
)
summarize_labels(rm3_analysis_20, f"BM25+RM3(fb={FB_TERMS})+TAS-B_gain_label")

,BM25+RM3(fb=10)+TAS-B_gain_label,count
0,neutral,41
1,helped,9
2,hurt,4


In [35]:
top_queries(rm3_analysis_20, f"BM25+RM3(fb={FB_TERMS})+TAS-B_gain", n=10, ascending=False)

,qid,query,BM25+RM3(fb=10)+TAS-B_gain,BM25+RM3(fb=10)+TAS-B,BM25+RM3(fb=20)+TAS-B,BM25+RM3(fb=30)+TAS-B,BM25+RM3(fb=5)+TAS-B,BM25+RM3(fb=50)+TAS-B,BM25+TAS-B,BM25+RM3(fb=5)+TAS-B_gain,BM25+RM3(fb=20)+TAS-B_gain,BM25+RM3(fb=30)+TAS-B_gain,BM25+RM3(fb=50)+TAS-B_gain,query_len_words,query_len_chars,has_colon,has_digit,is_question
14,1121353,what can you do about discrimination in the wo...,0.270007,0.825453,0.728026,0.823291,0.873232,0.823291,0.555447,0.317785,0.172579,0.267844,0.267844,12,70,False,False,False
29,174463,dog day afternoon meaning,0.261840,0.582539,0.582539,0.518043,0.582539,0.518043,0.320699,0.261840,0.261840,0.197344,0.197344,4,25,False,False,False
8,1108651,what the best way to get clothes white,0.213313,0.644030,0.686443,0.686443,0.665236,0.644030,0.430717,0.234520,0.255727,0.255727,0.213313,8,38,False,False,False
10,1110678,what is the un fao,0.080530,0.747478,0.747478,0.747478,0.747478,0.747478,0.666948,0.080530,0.080530,0.080530,0.080530,5,18,False,False,False
52,940547,when did rock n roll begin?,0.064210,0.611993,0.539587,0.539587,0.547782,0.539587,0.547782,0.000000,-0.008195,-0.008195,-0.008195,6,27,False,False,True
13,1116380,what is a nonconformity? earth science,0.051617,0.118177,0.118177,0.118177,0.154690,0.118177,0.066560,0.088130,0.051617,0.051617,0.051617,6,38,False,False,False
30,23849,are naturalization records public information,0.044983,0.432318,0.432318,0.432318,0.408021,0.432318,0.387335,0.020686,0.044983,0.044983,0.044983,5,45,False,False,False
53,997622,where is the show shameless filmed,0.044169,1.000000,1.000000,1.000000,1.000000,1.000000,0.955831,0.044169,0.044169,0.044169,0.044169,6,34,False,False,False
1,1037496,who is rep scalise?,0.042414,0.927449,0.927449,0.927449,0.927449,0.927449,0.885036,0.042414,0.042414,0.042414,0.042414,4,19,False,False,True
40,555530,what are best foods to lower cholesterol,0.009444,0.373260,0.373260,0.373260,0.338544,0.373260,0.363816,-0.025271,0.009444,0.009444,0.009444,7,40,False,False,False


In [36]:
top_queries(rm3_analysis_20, f"BM25+RM3(fb={FB_TERMS})+TAS-B_gain", n=10, ascending=True)

,qid,query,BM25+RM3(fb=10)+TAS-B_gain,BM25+RM3(fb=10)+TAS-B,BM25+RM3(fb=20)+TAS-B,BM25+RM3(fb=30)+TAS-B,BM25+RM3(fb=5)+TAS-B,BM25+RM3(fb=50)+TAS-B,BM25+TAS-B,BM25+RM3(fb=5)+TAS-B_gain,BM25+RM3(fb=20)+TAS-B_gain,BM25+RM3(fb=30)+TAS-B_gain,BM25+RM3(fb=50)+TAS-B_gain,query_len_words,query_len_chars,has_colon,has_digit,is_question
3,1051399,who sings monk theme song,-0.234054,0.563687,0.674954,0.706707,0.563687,0.706707,0.797741,-0.234054,-0.122786,-0.091033,-0.091033,5,25,False,False,False
42,640502,what does it mean if your tsh is low,-0.051279,0.863015,0.933746,0.933746,0.933746,0.933746,0.914294,0.019451,0.019451,0.019451,0.019451,9,36,False,False,False
2,1043135,who killed nicholas ii of russia,-0.038074,0.505473,0.505473,0.505473,0.505473,0.543548,0.543548,-0.038074,-0.038074,-0.038074,0.000000,6,32,False,False,False
43,67316,can fever cause miscarriage early pregnancy,-0.028745,0.293011,0.293011,0.293011,0.293011,0.293011,0.321756,-0.028745,-0.028745,-0.028745,-0.028745,6,43,False,False,False
28,169208,does mississippi have an income tax,0.000000,0.693266,0.693266,0.693266,0.693266,0.693266,0.693266,0.000000,0.000000,0.000000,0.000000,6,35,False,False,False
31,258062,how long does it take to remove wisdom tooth,0.000000,0.623554,0.692985,0.692985,0.692985,0.692985,0.623554,0.069431,0.069431,0.069431,0.069431,9,44,False,False,False
32,324585,how much money do motivational speakers make,0.000000,0.889954,0.889954,0.889954,0.889954,0.889954,0.889954,0.000000,0.000000,0.000000,0.000000,7,44,False,False,False
33,330975,how much would it cost to install my own wind ...,0.000000,0.621746,0.621746,0.621746,0.619628,0.621746,0.621746,-0.002118,0.000000,0.000000,0.000000,11,53,False,False,False
34,332593,how often to button quail lay eggs,0.000000,0.728769,0.728769,0.728769,0.728769,0.728769,0.728769,0.000000,0.000000,0.000000,0.000000,7,34,False,False,False
37,405163,is caffeine an narcotic,0.000000,0.420217,0.420217,0.420217,0.420217,0.420217,0.420217,0.000000,0.000000,0.000000,0.000000,4,23,False,False,False


In [37]:
compare_query_sets(
    rm3_analysis_20,
    label_column=f"BM25+RM3(fb={FB_TERMS})+TAS-B_gain_label",
    feature_columns=["query_len_words", "query_len_chars", "has_colon", "has_digit", "is_question"],
)

,BM25+RM3(fb=10)+TAS-B_gain_label,query_len_words,query_len_chars,has_colon,has_digit,is_question
0,helped,6.222222,34.888889,0.0,0.0,0.222222
1,hurt,6.500000,34.000000,0.0,0.0,0.000000
2,neutral,5.951220,33.512195,0.0,0.0,0.073171


In [38]:
rm3_metric_cols_20 = [f"BM25+RM3(fb={fb})+TAS-B" for fb in FB_VALUES]
rm3_gain_cols_20 = [f"BM25+RM3(fb={fb})+TAS-B_gain" for fb in FB_VALUES]
rm3_analysis_20[rm3_metric_cols_20].mean().sort_values(ascending=False)

BM25+RM3(fb=20)+TAS-B    0.676079
BM25+RM3(fb=30)+TAS-B    0.676006
BM25+RM3(fb=50)+TAS-B    0.675926
BM25+RM3(fb=5)+TAS-B     0.674037
BM25+RM3(fb=10)+TAS-B    0.671603
dtype: float64

In [39]:
rm3_best_20 = best_system_per_query(rm3_analysis_20, rm3_metric_cols_20)
summarize_best_systems(rm3_best_20)

,best_system,count
0,BM25+RM3(fb=5)+TAS-B,44
1,BM25+RM3(fb=10)+TAS-B,5
2,BM25+RM3(fb=20)+TAS-B,3
3,BM25+RM3(fb=50)+TAS-B,1
4,BM25+RM3(fb=30)+TAS-B,1


In [40]:
rm3_best_gain_20 = best_system_per_query(rm3_analysis_20, rm3_gain_cols_20, output_column="best_gain_setting")
summarize_best_systems(rm3_best_gain_20, best_system_column="best_gain_setting")

,best_gain_setting,count
0,BM25+RM3(fb=5)+TAS-B_gain,44
1,BM25+RM3(fb=10)+TAS-B_gain,5
2,BM25+RM3(fb=20)+TAS-B_gain,3
3,BM25+RM3(fb=50)+TAS-B_gain,1
4,BM25+RM3(fb=30)+TAS-B_gain,1


## Is the RM3 effect consistent per query on DL20?

Compare the gain from RM3 on the lexical pipeline with the gain from RM3 before TAS-B.

In [41]:
rm3_consistency_20 = perquery_20[["qid", "query", "BM25+RM3_gain"]].copy()
rm3_consistency_20["BM25+RM3+TAS-B_gain_vs_TAS-B"] = perquery_20["RM3_vs_TASB_gain"]

rm3_consistency_20["gain_rm3_bm25"] = rm3_consistency_20["BM25+RM3_gain"]
rm3_consistency_20["gain_rm3_rerank"] = rm3_consistency_20["BM25+RM3+TAS-B_gain_vs_TAS-B"]

rm3_consistency_20[["gain_rm3_bm25", "gain_rm3_rerank"]].corr()

,gain_rm3_bm25,gain_rm3_rerank
gain_rm3_bm25,1.000000,0.282482
gain_rm3_rerank,0.282482,1.000000


In [42]:
def gain_direction(value: float, threshold: float = 0.01) -> str:
    if value >= threshold:
        return "helped"
    if value <= -threshold:
        return "hurt"
    return "neutral"

rm3_consistency_20["bm25_effect"] = rm3_consistency_20["gain_rm3_bm25"].apply(gain_direction)
rm3_consistency_20["rerank_effect"] = rm3_consistency_20["gain_rm3_rerank"].apply(gain_direction)
rm3_consistency_20["consistency_group"] = (
    rm3_consistency_20["bm25_effect"] + " -> " + rm3_consistency_20["rerank_effect"]
)

rm3_consistency_20["consistency_group"].value_counts().rename_axis("consistency_group").reset_index(name="count")

,consistency_group,count
0,helped -> neutral,16
1,neutral -> neutral,15
2,hurt -> neutral,10
3,helped -> helped,5
4,hurt -> helped,2
5,hurt -> hurt,2
6,neutral -> helped,2
7,neutral -> hurt,1
8,helped -> hurt,1


In [43]:
rm3_consistency_20.sort_values("gain_rm3_bm25", ascending=False)[[
    "qid",
    "query",
    "gain_rm3_bm25",
    "gain_rm3_rerank",
    "consistency_group",
]].head(10)

,qid,query,gain_rm3_bm25,gain_rm3_rerank,consistency_group
17,1131069,how many sons robert kraft has,0.289308,0.000000,helped -> neutral
37,405163,is caffeine an narcotic,0.242716,0.000000,helped -> neutral
53,997622,where is the show shameless filmed,0.228123,0.044169,helped -> helped
22,1136962,why did the ancient egyptians call their land ...,0.135519,0.000000,helped -> neutral
28,169208,does mississippi have an income tax,0.128409,0.000000,helped -> neutral
10,1110678,what is the un fao,0.100964,0.080530,helped -> helped
48,877809,what metal are hip replacements made of,0.090589,0.000000,helped -> neutral
39,47210,average wedding dress alteration cost,0.080776,0.000000,helped -> neutral
21,1136047,difference between a company's strategy and bu...,0.076052,0.000000,helped -> neutral
29,174463,dog day afternoon meaning,0.069790,0.261840,helped -> helped


## `fb_terms` comparison on DL20

Compare the different `fb_terms` settings using `ndcg_cut_10` and `map`.

In [44]:
fb_terms_eval_20 = evaluate_runs(
    rm3_tasb_runs_20,
    topics_20_run,
    qrels_20_run,
    eval_metrics=["ndcg_cut_10", "map"],
)

fb_terms_table_20 = fb_terms_eval_20[fb_terms_eval_20["name"] != "BM25+TAS-B"].copy()
fb_terms_table_20["fb_terms"] = fb_terms_table_20["name"].str.extract(r"fb=(\d+)").astype(int)
fb_terms_table_20 = fb_terms_table_20[["fb_terms", "ndcg_cut_10", "map"]].sort_values("fb_terms")
fb_terms_table_20

,fb_terms,ndcg_cut_10,map
1,5,0.674037,0.415240
2,10,0.671603,0.412778
3,20,0.676079,0.414257
4,30,0.676006,0.415657
5,50,0.675926,0.416002


## BM25 + RM3 recall analysis on DL20

In [45]:
bm25_rm3_runs_20 = {"BM25": runs_20["BM25"]}

for fb in FB_VALUES:
    bm25_rm3_runs_20[f"BM25+RM3(fb={fb})"] = load_run(
        f"{CACHE_PREFIX}_bm25_rm3_fb{fb}_dl20.pkl"
    )

bm25_rm3_perquery_20_recall = perquery_comparison(
    bm25_rm3_runs_20,
    topics_20_run,
    qrels_20_run,
    metric="recall_100",
)

bm25_rm3_analysis_20_recall = add_gain_columns(
    bm25_rm3_perquery_20_recall,
    baseline="BM25",
    systems=[f"BM25+RM3(fb={fb})" for fb in FB_VALUES],
)
bm25_rm3_analysis_20_recall = add_query_features(bm25_rm3_analysis_20_recall)

In [46]:
bm25_rm3_metric_cols_20 = [f"BM25+RM3(fb={fb})" for fb in FB_VALUES]
bm25_rm3_gain_cols_20 = [f"BM25+RM3(fb={fb})_gain" for fb in FB_VALUES]
bm25_rm3_analysis_20_recall[bm25_rm3_metric_cols_20].mean().sort_values(ascending=False)

BM25+RM3(fb=5)     0.551599
BM25+RM3(fb=50)    0.551233
BM25+RM3(fb=30)    0.550948
BM25+RM3(fb=20)    0.549578
BM25+RM3(fb=10)    0.548754
dtype: float64

In [47]:
bm25_rm3_best_20 = best_system_per_query(bm25_rm3_analysis_20_recall, bm25_rm3_metric_cols_20)
summarize_best_systems(bm25_rm3_best_20)

,best_system,count
0,BM25+RM3(fb=5),34
1,BM25+RM3(fb=20),7
2,BM25+RM3(fb=10),7
3,BM25+RM3(fb=30),4
4,BM25+RM3(fb=50),2


In [48]:
bm25_rm3_best_gain_20 = best_system_per_query(
    bm25_rm3_analysis_20_recall,
    bm25_rm3_gain_cols_20,
    output_column="best_gain_setting",
)
summarize_best_systems(bm25_rm3_best_gain_20, best_system_column="best_gain_setting")

,best_gain_setting,count
0,BM25+RM3(fb=5)_gain,34
1,BM25+RM3(fb=20)_gain,7
2,BM25+RM3(fb=10)_gain,7
3,BM25+RM3(fb=30)_gain,4
4,BM25+RM3(fb=50)_gain,2


In [49]:
bm25_rm3_analysis_20_recall = label_gain(
    bm25_rm3_analysis_20_recall,
    gain_column=f"BM25+RM3(fb={FB_TERMS})_gain",
)
summarize_labels(bm25_rm3_analysis_20_recall, f"BM25+RM3(fb={FB_TERMS})_gain_label")

,BM25+RM3(fb=10)_gain_label,count
0,neutral,26
1,helped,21
2,hurt,7


In [50]:
top_queries(bm25_rm3_analysis_20_recall, f"BM25+RM3(fb={FB_TERMS})_gain", n=10, ascending=False)

,qid,query,BM25+RM3(fb=10)_gain,BM25,BM25+RM3(fb=10),BM25+RM3(fb=20),BM25+RM3(fb=30),BM25+RM3(fb=5),BM25+RM3(fb=50),BM25+RM3(fb=5)_gain,BM25+RM3(fb=20)_gain,BM25+RM3(fb=30)_gain,BM25+RM3(fb=50)_gain,query_len_words,query_len_chars,has_colon,has_digit,is_question
13,1116380,what is a nonconformity? earth science,0.500000,0.062500,0.562500,0.562500,0.562500,0.625000,0.562500,0.562500,0.500000,0.500000,0.500000,6,38,False,False,False
47,768208,what is mamey,0.465116,0.441860,0.906977,0.790698,0.860465,0.930233,0.837209,0.488372,0.348837,0.418605,0.395349,3,13,False,False,False
46,730539,what is chronometer who invented it,0.384615,0.576923,0.961538,0.961538,0.961538,0.961538,0.961538,0.384615,0.384615,0.384615,0.384615,6,35,False,False,False
1,1037496,who is rep scalise?,0.271605,0.518519,0.790123,0.691358,0.753086,0.802469,0.740741,0.283951,0.172840,0.234568,0.222222,4,19,False,False,True
7,1106979,define pareto chart in statistics,0.196970,0.636364,0.833333,0.863636,0.863636,0.848485,0.863636,0.212121,0.227273,0.227273,0.227273,5,33,False,False,False
10,1110678,what is the un fao,0.168675,0.506024,0.674699,0.686747,0.686747,0.698795,0.686747,0.192771,0.180723,0.180723,0.180723,5,18,False,False,False
53,997622,where is the show shameless filmed,0.117647,0.470588,0.588235,0.558824,0.529412,0.647059,0.529412,0.176471,0.088235,0.058824,0.058824,6,34,False,False,False
29,174463,dog day afternoon meaning,0.117021,0.127660,0.244681,0.234043,0.223404,0.265957,0.223404,0.138298,0.106383,0.095745,0.095745,4,25,False,False,False
8,1108651,what the best way to get clothes white,0.113043,0.113043,0.226087,0.243478,0.226087,0.260870,0.208696,0.147826,0.130435,0.113043,0.095652,8,38,False,False,False
18,1132532,average annual income data analyst,0.072727,0.290909,0.363636,0.345455,0.336364,0.327273,0.345455,0.036364,0.054545,0.045455,0.054545,5,34,False,False,False
